In [42]:
import numpy as np
import pandas as pd
import pickle


from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score

In [43]:
df_train = pd.read_csv(r'../data/cleaned/train.csv')
df_test = pd.read_csv(r'../data/cleaned/test.csv')
df_val = pd.read_csv(r'../data/cleaned/val.csv')

In [44]:
df_train

,Datetime,PJME_MW,year,month,hour,day_num,day_of_month,week_of_year,is_weekend,lag_1,lag_24,lag_168,rolling_mean_24,rolling_std_24
0,2002-01-09 01:00:00,0.306289,0.0,0.000000,0.043478,0.333333,0.266667,0.019231,0,0.345497,0.313937,0.286042,0.456556,0.234076
1,2002-01-09 02:00:00,0.286738,0.0,0.000000,0.086957,0.333333,0.266667,0.019231,0,0.306289,0.297609,0.271632,0.456097,0.236377
2,2002-01-09 03:00:00,0.279890,0.0,0.000000,0.130435,0.333333,0.266667,0.019231,0,0.286738,0.291394,0.268766,0.455444,0.240153
3,2002-01-09 04:00:00,0.278416,0.0,0.000000,0.173913,0.333333,0.266667,0.019231,0,0.279890,0.294912,0.273654,0.454754,0.244299
4,2002-01-09 05:00:00,0.289982,0.0,0.000000,0.217391,0.333333,0.266667,0.019231,0,0.278416,0.310060,0.292026,0.453764,0.250095
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101635,2013-08-13 20:00:00,0.560813,1.0,0.636364,0.869565,0.166667,0.400000,0.615385,0,0.592205,0.599199,0.419973,0.532663,0.430098
101636,2013-08-13 21:00:00,0.558074,1.0,0.636364,0.913043,0.166667,0.400000,0.615385,0,0.560813,0.590393,0.431118,0.530359,0.422228
101637,2013-08-13 22:00:00,0.517455,1.0,0.636364,0.956522,0.166667,0.400000,0.615385,0,0.558074,0.551796,0.412262,0.528420,0.415776
101638,2013-08-13 23:00:00,0.449342,1.0,0.636364,1.000000,0.166667,0.400000,0.615385,0,0.517455,0.477594,0.363194,0.526359,0.411341


In [45]:
with open('target_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [46]:
#calculation metrices
def evaluate_forecast(y_true, y_pred):
    
    y_pred= np.asarray(y_pred).reshape(-1, 1)
    y_true= np.asarray(y_true).reshape(-1, 1)

    y_pred = scaler.inverse_transform(y_pred)
    y_true= scaler.inverse_transform(y_true)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return mae, rmse, mape, r2

In [47]:
#navie forecast 

"""" it the consuptional assumption value in navie force cast we assume that yesterday usage will be today uage"""

'" it the consuptional assumption value in navie force cast we assume that yesterday usage will be today uage'

In [48]:
test_df = df_test.copy()

test_df['pred_naive'] = df_test['lag_1']

test_df['pred_prev_day'] = df_test['lag_24']

test_df['pred_prev_week'] = df_test['lag_168']

test_df['pred_moving_avg'] = df_test['rolling_mean_24']

In [49]:
baselines = {
    'Naive (Lag 1)': test_df['pred_naive'],
    'Previous-Day (Lag 24)': test_df['pred_prev_day'],
    'Previous-Week (Lag 168)': test_df['pred_prev_week'],
    '24-Hour Moving Average': test_df['pred_moving_avg']
}

In [50]:
results = []
for name, preds in baselines.items():
    mae, rmse, mape, r2 = evaluate_forecast(test_df['PJME_MW'], preds)
    results.append({
        'Model': name,
        'MAE (MW)': round(mae, 2),
        'RMSE (MW)': round(rmse, 2),
        'R2 score': round(r2,2),
        'MAPE (%)': round(mape, 2)
    })

In [51]:
results

[{'Model': 'Naive (Lag 1)',
  'MAE (MW)': 1054.66,
  'RMSE (MW)': np.float64(1354.57),
  'R2 score': 0.96,
  'MAPE (%)': np.float64(3.43)},
 {'Model': 'Previous-Day (Lag 24)',
  'MAE (MW)': 2206.48,
  'RMSE (MW)': np.float64(3027.62),
  'R2 score': 0.78,
  'MAPE (%)': np.float64(7.01)},
 {'Model': 'Previous-Week (Lag 168)',
  'MAE (MW)': 3434.72,
  'RMSE (MW)': np.float64(4735.51),
  'R2 score': 0.46,
  'MAPE (%)': np.float64(10.7)},
 {'Model': '24-Hour Moving Average',
  'MAE (MW)': 3946.08,
  'RMSE (MW)': np.float64(5038.26),
  'R2 score': 0.39,
  'MAPE (%)': np.float64(13.13)}]

In [52]:
df_base = pd.DataFrame(results)

In [53]:
df_base

,Model,MAE (MW),RMSE (MW),R2 score,MAPE (%)
0,Naive (Lag 1),1054.66,1354.57,0.96,3.43
1,Previous-Day (Lag 24),2206.48,3027.62,0.78,7.01
2,Previous-Week (Lag 168),3434.72,4735.51,0.46,10.70
3,24-Hour Moving Average,3946.08,5038.26,0.39,13.13
